# Bolånerisk-kalkylator
Interaktivt verktyg för att bedöma risken att en låntagare slutar betala sitt bolån, med hjälp av en tränad logistisk regressionsmodell (Gini: 0.426).

**Användning:** Justera sliders för att simulera olika låntagarprofiler. Modellen beräknar risken att en låntagare slutar betala i realtid.

| Risknivå | Risken att en låntagare slutar betala | Rekommendation |
|---|---|---|
| 🟢 Låg | < 5% | Godkänn |
| 🟡 Medel | 5–20% | Manuell granskning |
| 🔴 Hög | > 20% | Avslå |

In [14]:
import pickle #pickle är ett inbyggt Python-bibliotek för att spara och ladda Python-objekt till fil. 
#Den sparar den tränade modellen till en fil — annars försvinner den när man stänger notebooken och man måste träna om från scratch varje gång.
#Hämtar tillbaka den sparade modellen så kalkylatorn kan använda den utan att träna om.
import numpy as np
import pandas as pd
import ipywidgets as widgets #Det är verktygslådan som innehåller alla interaktiva kontroller. Utan ipywidgets skulle kalkylatorn bara vara vanlig kod där man manuellt ändrar siffror 
# — med ipywidgets blir det ett interaktivt verktyg med sliders.
from IPython.display import display, clear_output #display: Visar ett objekt i notebooken — i detta fall sliders och outputfältet. Utan display skulle sliders inte synas alls.
#clear_output: Rensar det som visas i outputfältet — används i beräkningsfunktionen. Varje gång man drar i en slider rensas det gamla resultatet bort och ersätts med det nya. 
#Utan clear_output skulle resultaten staplas på varandra — man skulle se hundratals gamla riskbedömningar under varandra istället för bara den senaste.

# Hämtar den tränade modellen och scalern som sparades i bolan_analys.ipynb — utan dessa vet kalkylatorn ingenting.
with open('bolan_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

#Skapar fem reglage — ett för varje variabel modellen behöver. value är startvärdet, min och max är gränserna man kan dra mellan.
#skuldkvot = total skuld / inkomst — alla skulder låntagaren har, inte bara bolånet
                                    #startar på 4.5 (lånet är 4.5 gånger årsinkomsten)
skuldkvot     = widgets.FloatSlider(value=4.5, min=1.0, max=10.0, step=0.1, description='Skuldkvot:')
                                    #startar på 15% minst 10% (lägsta tillåtna i Sverige)
kontantinsats = widgets.IntSlider(value=15, min=10, max=90, description='Kontant%:')
#lån/inkomst = just detta bolån / inkomst — bara det specifika lånet vi bedömer
                                    #startar på 3.0 (lånet är 3 gånger årsinkomsten)
lan_inkomst   = widgets.FloatSlider(value=3.0, min=1.0, max=10.0, step=0.1, description='Lån/inkomst:')
                                    #startar på 8 000 kr/månad
manadskostnad = widgets.IntSlider(value=8000, min=1000, max=30000, step=500, description='Mån.kostn:')
                                    #startar på 3.5% ränta, vilket är ungefär dagens nivå
ranta         = widgets.FloatSlider(value=3.5, min=1.0, max=7.0, step=0.1, description='Ränta%:')
#I en verklig modell skulle skuldkvoten inkludera alla låntagarens skulder — billån, privatlån och så vidare — men i mitt projekt hade jag bara bolånsdata så de blev nästan 
# identiska. I produktion med riktig bankdata skulle de skilja sig mer.
output        = widgets.Output()

#Denna funktion körs automatiskt varje gång man drar i en slider. Den: Samlar ihop de fem slider-värdena i en DataFrame
#Skalar om värdena med StandardScaler #Ber modellen beräkna sannolikheten för betalningsproblem
#Multiplicerar med 100 för att få procent #Skriver ut risknivå och rekommendation
def berakna(change=None):
    with output:
        clear_output(wait=True)
        features = pd.DataFrame(
            [[skuldkvot.value, kontantinsats.value, lan_inkomst.value, manadskostnad.value, ranta.value]],
            columns=['skuldkvot', 'kontantinsats_procent', 'lan_till_inkomst', 'manadskostnad', 'ranta']
        )
        features_scaled = scaler.transform(features)
        risk = model.predict_proba(features_scaled)[0][1] * 100

        if risk < 5:
            status = '🟢 Låg risk — Rekommenderas'
        elif risk < 20:
            status = '🟡 Medel risk — Manuell granskning'
        else:
            status = '🔴 Hög risk — Avslå'

        print(f'Risken att en låntagare slutar betala: {risk:.1f}%')
        print(f'Modell-Gini:  0.426')
        print(f'Status:       {status}')
        print('OBS: Riksdagen röstar 4 mars 2026 om att sänka kravet på kontantinsats från 15% till 10%')

#Säger åt varje slider — "när ditt värde ändras, kör berakna automatiskt."
for w in [skuldkvot, kontantinsats, lan_inkomst, manadskostnad, ranta]:
    w.observe(berakna, names='value')

#Visar sliders och outputfältet i notebooken, och kör beräkningen en gång direkt så resultatet syns redan från start.
display(skuldkvot, kontantinsats, lan_inkomst, manadskostnad, ranta, output)
berakna()

FloatSlider(value=4.5, description='Skuldkvot:', max=10.0, min=1.0)

IntSlider(value=15, description='Kontant%:', max=90, min=10)

FloatSlider(value=3.0, description='Lån/inkomst:', max=10.0, min=1.0)

IntSlider(value=8000, description='Mån.kostn:', max=30000, min=1000, step=500)

FloatSlider(value=3.5, description='Ränta%:', max=7.0, min=1.0)

Output()